<a href="https://colab.research.google.com/github/JIGGISTYLE/handwritten/blob/main/handwritten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#workflow to convert image into eq


Upload image

      ↓
Convert to grayscale

      ↓
Threshold (black/white)

      ↓
Find contours

      ↓
Segment characters

      ↓
Show detected characters


In [ ]:
import cv2
import matplotlib.pyplot as plt

# load image
# img_path = "/content/drive/MyDrive/projects/equation_project/test_images/equation1.jpg"
image = cv2.imread("eq1.jpeg")

# convert BGR to RGB , because opencv user bgr as defaut and others as rgb
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.imshow(image)
plt.title("Uploaded Image")
plt.axis("off")
plt.show()



In [ ]:
# covert to gray
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")
plt.show()

In [ ]:
# covert to black and white / thrasholding image / binary image
_, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)  # cv2,threshold(gray_img, thresh, max_value , thresholding type) it returns rit(thresh , not important),thresh_img

plt.imshow(thresh, cmap="gray")
plt.title("Threshold")
plt.axis("off")
plt.show()

In [ ]:
# detect charcchters or contours
contours, _ = cv2.findContours(
    thresh,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)


In [ ]:
# bounding boxex . visulaizezs the detected charchteds
import numpy as np

image_copy = image.copy()

boxes = []

for cnt in contours:

    x,y,w,h = cv2.boundingRect(cnt)

    if w*h > 100:  # to remove dust or very small contures

      boxes.append((x,y,w,h))

    cv2.rectangle(image_copy,(x,y),(x+w,y+h),(255,0,0),2)

plt.imshow(image_copy)
plt.title("Detected Characters")
plt.axis("off")
plt.show()


In [ ]:
# sort boxes
boxes = sorted(boxes, key=lambda a: a[0])  # from left to right , sort x coordinate wise ,  a varible is a[0] of (x,y,w,h)
boxes

In [ ]:
#crop each charchter
characters = []

for (x, y, w, h) in boxes:
    char = thresh[y:y+h, x:x+w]   # numpy slicing its image[row,colomn] not image[x,y], crop from threshold image , from y to y+height , and x to x+width
    characters.append(char)


In [ ]:
#resize
resized_chars = []

for char in characters:
    char = cv2.resize(char, (28, 28))
    resized_chars.append(char)



In [ ]:
# normalize for cnn
normalized_chars = []

for char in resized_chars:
    char = char / 255.0
    normalized_chars.append(char)


In [ ]:
# convert to tensors
import torch

tensor_chars = []

for char in normalized_chars:
    char = np.array(char, dtype=np.float32)
    char = np.expand_dims(char, axis=0)   # add channel dim → (1,28,28)
    char = torch.tensor(char)
    tensor_chars.append(char)



In [ ]:
# make batches
input_batch = torch.stack(tensor_chars)
input_batch.size()

In [ ]:

pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# cnnmodel
class CharCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # [N,32,14,14]
        x = self.pool(F.relu(self.conv2(x)))   # [N,64,7,7]

        x = x.view(x.size(0), -1)              # flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
labels = ['0','1','2','3','4','5','6','7','8','9','+','-','*','/','=','x','y']

In [ ]:
num_classes = len(labels)

In [ ]:
model = CharCNN(num_classes)
model.eval()

now we will make , make the model from dataset to final prediction

use triple ` for structures

```
MyDrive/
└── equation_project/
    └── dataset/
        ├── train/
        │   ├── 0/
        │   ├── 1/
        │   ├── 2/
        │   ├── ...
        │   ├── plus/
        │   ├── minus/
        │   ├── multiply/
        │   ├── divide/
        │   ├── equal/
        │   ├── x/
        │   └── y/
        │
        └── test/
            ├── 0/
            ├── 1/
            ├── ...
            ├── plus/
            ├── minus/
            ├── multiply/
            ├── divide/
            ├── equal/
            ├── x/
            └── y/
```

In [ ]:
!git add .
!git commit -m "Daily update"
!git push

In [ ]:
# organize collab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_path = "/content/drive/MyDrive/projects/equation_project"
os.makedirs(project_path, exist_ok=True)

print("Project folder ready")

In [ ]:
base = "/content/drive/MyDrive/projects/equation_project/dataset"

splits = ["train", "test"]
classes = ['0','1','2','3','4','5','6','7','8','9',
           'plus','minus','multiply','divide','equal',
           'x','y','a','b']

for split in splits:
    for cls in classes:
        os.makedirs(f"{base}/{split}/{cls}", exist_ok=True)

print("Dataset folders created")

In [ ]:
# start downloading
import os
import cv2
from torchvision import datasets

In [ ]:
mnist_train = datasets.MNIST(root="/content/data", train=True, download=True)
mnist_test  = datasets.MNIST(root="/content/data", train=False, download=True)

In [ ]:
base = "/content/drive/MyDrive/projects/equation_project/dataset"

In [ ]:
import numpy as np
import cv2

def save_mnist_fast(dataset, split, limit_per_class):
    counts = {str(i): 0 for i in range(10)}

    for img, label in dataset:
        cls = str(label)

        if counts[cls] >= limit_per_class:
            continue

        img = np.array(img)
        path = f"{base}/{split}/{cls}/{counts[cls]}.png"
        cv2.imwrite(path, img)

        counts[cls] += 1

        if all(v >= limit_per_class for v in counts.values()):
            break

    print(f"{split} saved")

In [ ]:
# save_mnist_fast(mnist_test, "test", 500)

In [ ]:

base = "/content/drive/MyDrive/projects/equation_project/dataset"


In [ ]:
import os

train_path = base +"/train"

for cls in sorted(os.listdir(train_path)):
    cls_path = os.path.join(train_path, cls)
    print(cls, "→", len(os.listdir(cls_path)))

In [ ]:
import os

test_path = base +"/test"

for cls in sorted(os.listdir(test_path)):
    cls_path = os.path.join(test_path, cls)
    print(cls, "→", len(os.listdir(cls_path)))

In [ ]:
# emnist for letter a b x y
import torchvision
from torchvision import datasets, transforms

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
train_data = datasets.EMNIST(
    root="emnist_data",
    split="letters",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.EMNIST(
    root="emnist_data",
    split="letters",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
target = {
    1: 'a',
    2: 'b',
    24: 'x',
    25: 'y'
}

In [ ]:
import os
from PIL import Image
import numpy as np

base = "/content/drive/MyDrive/projects/equation_project/dataset"

def save_emnist_torch(dataset, split):
    for i in range(len(dataset)):
        img, label = dataset[i]

        if label in target:
            char = target[label]
            folder = f"{base}/{split}/{char}"
            os.makedirs(folder, exist_ok=True)

            img = img.squeeze().numpy() * 255
            img = img.astype(np.uint8)

            # Fix EMNIST orientation
            img = np.rot90(img, k=3)
            img = np.fliplr(img)

            Image.fromarray(img).save(f"{folder}/{i}.png")

In [ ]:
for ch in ['a','b','x','y']:
    print(ch, "train:", len(os.listdir(f"{base}/train/{ch}")))
    print(ch, "test :", len(os.listdir(f"{base}/test/{ch}")))

In [ ]:
# for symbols
!wget https://zenodo.org/record/259444/files/HASYv2.tar.bz2
!tar -xjf HASYv2.tar.bz2

In [ ]:
needed_symbols = {
    '+': 'plus',
    '-': 'minus',
    '*': 'times',
    '/': 'division',
    '=': 'equals',
    '(': 'left_parenthesis',
    ')': 'right_parenthesis'
}

In [ ]:
import pandas as pd

symbols_df = pd.read_csv("HASYv2/symbols.csv")
symbols_df.head()